In [2]:
!pip install -q langgraph langchain langchain-google-genai

In [10]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

print("API Loaded")

API Loaded


In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.1
)

In [12]:
response = llm.invoke("Say hello")

print(response.content)

Hello!


In [13]:
from typing import TypedDict


In [14]:
class AgentState(TypedDict):
    problem: str
    analysis: str
    generated_code: str
    solved: bool

In [16]:
def analyze_problem(state):

    prompt = f"""
You are a competitive programming expert.

Analyze this problem briefly.

Problem:
{state["problem"]}

Give:
1. Key observation
2. Algorithm idea
"""

    response = llm.invoke(prompt)

    state["analysis"] = response.content

    return state

In [32]:
def generate_code(state):

    prompt = f"""
You are a competitive programming expert.

Problem:
{state["problem"]}

Analysis:
{state["analysis"]}

Generate ONLY Python code.

No explanation.
"""

    response = llm.invoke(prompt)

    state["generated_code"] = (
    response.content
    .replace("```python", "")
    .replace("```", "")
    .strip()
)

    return state

In [18]:
def evaluate_code(state):

    prompt = f"""
You are a competitive programming judge.

Problem:
{state["problem"]}

Code:
{state["generated_code"]}

Is this solution logically correct?

Answer only:

YES

or

NO
"""

    response = llm.invoke(prompt)

    verdict = response.content.strip()

    state["solved"] = "YES" in verdict.upper()

    return state

In [33]:
def improve_code(state):

    prompt = f"""
You are an expert competitive programmer.

Problem:
{state["problem"]}

Previous Code:
{state["generated_code"]}

The previous solution may contain mistakes.

Generate an improved Python solution.

Return ONLY Python code.
Do not explain.
Do not use markdown.
"""

    response = llm.invoke(prompt)

    state["generated_code"] = (
    response.content
    .replace("```python", "")
    .replace("```", "")
    .strip()
)

    return state

In [34]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

In [35]:
workflow.add_node("analyze", analyze_problem)
workflow.add_node("generate", generate_code)
workflow.add_node("evaluate", evaluate_code)
workflow.add_node("improve", improve_code)

In [36]:
print("workflow" in globals())

True


In [37]:
def should_continue(state):

    if state["solved"]:
        return "end"

    return "improve"

In [38]:
workflow.set_entry_point("analyze")

workflow.add_edge("analyze", "generate")
workflow.add_edge("generate", "evaluate")

In [39]:
workflow.add_conditional_edges(
    "evaluate",
    should_continue,
    {
        "end": END,
        "improve": "improve"
    }
)

workflow.add_edge("improve", "evaluate")

In [40]:
graph = workflow.compile()

print("Graph compiled successfully!")

Graph compiled successfully!


# Problem 1 Halloumi Boxes

In [41]:
state = {
    "problem": """
A. Halloumi Boxes

Theofanis is busy after his last contest, as now, he has to deliver many halloumis all over the world. He stored them inside n boxes and each box has some number ai written on it.

He wants to sort them in non-decreasing order by their number, however, his machine works in a strange way. It can only reverse any subarray of boxes with length at most k.

Find if it's possible to sort the boxes using any number of reverses.

Input:
The first line contains t — the number of test cases.

The first line of each test case contains two integers n and k.

The second line contains n integers a1, a2, ..., an.

Output:
For each test case print YES if the array can be sorted, otherwise NO.
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [42]:
result = graph.invoke(state)

In [43]:
print(result["generated_code"])

def solve():
    n, k = map(int, input().split())
    a = list(map(int, input().split()))

    if k == 1:
        # If k=1, only reversing length 1 subarrays is allowed, which changes nothing.
        # So, the array can only be sorted if it's already sorted.
        is_sorted = True
        for i in range(n - 1):
            if a[i] > a[i+1]:
                is_sorted = False
                break
        if is_sorted:
            print("YES")
        else:
            print("NO")
    else:
        # If k >= 2, we can reverse any subarray of length 2.
        # Reversing [x, y] results in [y, x], effectively swapping adjacent elements.
        # The ability to swap any two adjacent elements is sufficient to sort any array
        # (e.g., using bubble sort logic). Since we can perform any number of reverses,
        # we can always sort the array.
        print("YES")

t = int(input())
for _ in range(t):
    solve()


In [44]:
print(result["solved"])

True


In [45]:
print(result)

{'problem': "\nA. Halloumi Boxes\n\nTheofanis is busy after his last contest, as now, he has to deliver many halloumis all over the world. He stored them inside n boxes and each box has some number ai written on it.\n\nHe wants to sort them in non-decreasing order by their number, however, his machine works in a strange way. It can only reverse any subarray of boxes with length at most k.\n\nFind if it's possible to sort the boxes using any number of reverses.\n\nInput:\nThe first line contains t — the number of test cases.\n\nThe first line of each test case contains two integers n and k.\n\nThe second line contains n integers a1, a2, ..., an.\n\nOutput:\nFor each test case print YES if the array can be sorted, otherwise NO.\n", 'analysis': 'Here\'s a brief analysis of the problem:\n\n1.  **Key Observation:**\n\n    The crucial insight lies in understanding the power of the "reverse subarray of length at most k" operation based on the value of `k`.\n\n    *   **Case 1: `k = 1`**\n    

# problem 2 Line Trip

In [46]:
state = {
    "problem": """
A. Line Trip

There is a road, which can be represented as a number line. You are located at the point 0 of the number line, and you want to travel from the point 0 to the point x, and back to the point 0.

You travel by car, which spends 1 liter of gasoline per 1 unit of distance travelled. When you start at the point 0, your car is fully fueled (its gas tank contains the maximum possible amount of fuel).

There are n gas stations, located at points a1,a2,...,an. When you arrive at a gas station, you fully refuel your car. Note that you can refuel only at gas stations, and there are no gas stations in points 0 and x.

Input:
...
Output:
...

""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [47]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

Solved: True
def solve():
    n, x = map(int, input().split())
    a = list(map(int, input().split()))

    max_required_fuel = 0

    # Segment from 0 to the first gas station a[0]
    max_required_fuel = max(max_required_fuel, a[0] - 0)

    # Segments between consecutive gas stations
    for i in range(n - 1):
        max_required_fuel = max(max_required_fuel, a[i+1] - a[i])

    # Segment from the last gas station a[n-1] to x and back to a[n-1]
    # This requires 2 * (x - a[n-1]) fuel
    max_required_fuel = max(max_required_fuel, 2 * (x - a[n-1]))

    print(max_required_fuel)

if __name__ == '__main__':
    solve()


# Problem 3 Cover in Water

In [48]:
state = {
    "problem": """
A. Cover in Water

Flip has a row of cells, some of which are blocked, and some are empty. He wants all empty cells to have water in them. He has two actions at his disposal:

1 - place water in an empty cell.
2 - remove water from a cell and place it in any other empty cell.

If at some moment cell i (2 <= i <= n - 1) is empty and both cells i - 1 and i + 1 contain water, then it becomes filled with water.

Find the minimum number of times he needs to perform action 1 in order to fill all empty cells with water.

Note that you don't need to minimize the use of action 2. Note that blocked cells neither contain water nor can Flip place water in them.

Input

Each test contains multiple test cases. The first line contains the number of test cases t (1 <= t <= 100). The description of the test cases follows.

The first line of each test case contains a single integer n (1 <= n <= 100) — the number of cells.

The next line contains a string s of length n. The i-th character of s is '.' if the cell i is empty and '#' if cell i is blocked.

Output

For each test case, output a single number — the minimal amount of actions 1 needed to fill all empty cells with water.

Example

Input
5
3
...
7
##..#.#
7
#######
10
.#..#...#.
3
...

Output
2
2
0
2
2
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [50]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

KeyboardInterrupt: 

In [79]:
# I skipped it because it was taking too much time.

# Problem 4 Game with Integers

In [52]:
state = {
    "problem": """
A. Game with Integers

Vanya and Vova are playing a game. Players are given an integer n. On their turn, the player can add 1 to the current integer or subtract 1. The players take turns; Vanya starts. If after Vanya's move the integer is divisible by 3, then he wins. If 10 moves have passed and Vanya has not won, then Vova wins.

Write a program that determines who will win if both players play optimally.

Input

The first line contains the integer t (1 <= t <= 100) — the number of test cases.

The single line of each test case contains the integer n (1 <= n <= 1000).

Output

For each test case print "First" if Vanya wins and "Second" if Vova wins.

Example

Input
6
1
3
5
100
999
1000

Output
First
Second
First
First
Second
First
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [53]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

Solved: True
def solve():
    n = int(input())
    if n % 3 == 0:
        print("Second")
    else:
        print("First")

t = int(input())
for _ in range(t):
    solve()


# Problem 5 Coins

In [55]:
state = {
    "problem": """
A. Coins

In Berland, there are two types of coins, having denominations of 2 and k burles.

Determine whether it is possible to represent n burles in coins.

Input

The first line contains a single integer t (1 <= t <= 10^4).

The only line of each test case contains two integers n and k.

Output

For each test case print YES if it is possible to represent n burles in coins; otherwise print NO.

Example

Input
4
5 3
6 1
7 4
8 8

Output
YES
YES
NO
YES
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [56]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

Solved: True
import sys

def solve():
    n, k = map(int, sys.stdin.readline().split())

    if n % 2 == 0:
        # If n is even, we can always form it using only 2-burle coins.
        # For example, n = 2 * (n // 2) + k * 0
        print("YES")
    else:
        # If n is odd:
        if k % 2 == 0:
            # If k is even, then 2x is even and ky (even * y) is also even.
            # The sum 2x + ky will always be even.
            # So, an odd n cannot be formed.
            print("NO")
        else:
            # If k is odd:
            # We need to use an odd number of k-burle coins to make ky odd.
            # The smallest positive odd number of k-burle coins is 1.
            # If we use one k-burle coin, the remaining amount is n - k.
            # For this to be possible:
            # 1. n - k must be non-negative (n >= k).
            # 2. n - k must be even (so it can be formed by 2-burle coins).
            # Since n is odd and k is odd, n - k is (odd - odd) = even

In [60]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY_2")

In [61]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.1
)

In [62]:
response = llm.invoke("hello")
print(response.content)

Hello! How can I help you today?


# Problem 6 Extremly Round

In [68]:
state = {
    "problem": """
A. Extremely Round

Let's call a positive integer extremely round if it has only one non-zero digit.

You are given an integer n. Calculate the number of extremely round integers x such that 1 <= x <= n.

Input

The first line contains one integer t (1 <= t <= 10^4).

Then t lines follow. The i-th of them contains one integer n (1 <= n <= 999999).

Output

For each test case, print one integer — the number of extremely round integers x such that 1 <= x <= n.

Example

Input
5
9
42
13
100
111

Output
9
13
10
19
19
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [64]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

Solved: True
def solve():
    n = int(input())
    
    count = 0
    power_of_10 = 1
    
    while power_of_10 <= n:
        # For the current power_of_10 (e.g., 1, 10, 100, ...),
        # we count how many multiples (1*p, 2*p, ..., 9*p) are <= n.
        # The maximum digit 'd' such that d * power_of_10 <= n is n // power_of_10.
        # Since 'd' must be between 1 and 9, we take the minimum of 9 and this value.
        count += min(9, n // power_of_10)
        
        # Move to the next power of 10
        power_of_10 *= 10
        
    print(count)

t = int(input())
for _ in range(t):
    solve()


# Promble 7 Goals of Victor

In [66]:
state = {
    "problem": """
A. Goals of Victory

There are n teams in a football tournament. Each pair of teams match up once.

After every match, Pack Chanek receives two integers as the result of the match, the number of goals the two teams scored during the match.

The efficiency of a team is equal to the total number of goals the team scores in each of its matches minus the total number of goals scored by the opponent in each of its matches.

Given the efficiency of n−1 teams, determine the efficiency of the missing team.

Input:
The first line contains an integer t (1 ≤ t ≤ 500).

For each test case:
The first line contains an integer n (2 ≤ n ≤ 100).
The second line contains n−1 integers a1, a2, ..., a(n−1).

Output:
For each test case, output the efficiency of the missing team.
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [67]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

Solved: True
def solve():
    n = int(input())
    a = list(map(int, input().split()))

    sum_of_given_efficiencies = sum(a)
    
    missing_efficiency = -sum_of_given_efficiencies
    print(missing_efficiency)

t = int(input())
for _ in range(t):
    solve()


# Proble 8 Walking Master

In [69]:
state = {
    "problem": """
A. Walking Master

YunQian is standing on an infinite plane with the Cartesian coordinate system on it.

In one move, she can move to the diagonally adjacent point on the top right or the adjacent point on the left.

That is, if she is standing on point (x, y), she can either move to point (x + 1, y + 1) or point (x - 1, y).

Initially she stands at point (a, b) and wants to move to point (c, d).

Find the minimum number of moves she needs to make or determine that it is impossible.

Input:
The first line contains an integer t.

Each test case contains four integers a, b, c, d.

Output:
For each test case, if it is possible to move from point (a, b) to point (c, d), output the minimum number of moves. Otherwise, output -1.
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [70]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

Solved: True
def solve():
    a, b, c, d = map(int, input().split())

    if d < b:
        print(-1)
        return

    # Number of Top Right moves needed to reach target y-coordinate
    n_tr = d - b

    # Current x-coordinate after n_tr Top Right moves
    # Each TR move increases x by 1
    current_x = a + n_tr

    if current_x < c:
        print(-1)
        return

    # Number of Left moves needed to reach target x-coordinate
    # Each L move decreases x by 1
    n_l = current_x - c

    print(n_tr + n_l)

t = int(input())
for _ in range(t):
    solve()


# Proble 9 Grasshopper on a Line

In [71]:
state = {
    "problem": """
A. Grasshopper on a Line

You are given two integers x and k.

Grasshopper starts at point 0 on the number line.

In one move, it can jump some integer distance that is NOT divisible by k.

Find the minimum number of moves needed to reach point x.

If there are multiple answers, output any.

Input:
The first line contains t.

Each test case contains two integers x and k.

Output:
First print the minimum number of moves.

Then print the jumps themselves.

Each jump must not be divisible by k.

The final position must be exactly x.
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [72]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

Solved: True
The problem asks us to find the minimum number of moves a grasshopper needs to reach point `x` starting from point 0. In one move, the grasshopper can jump any integer distance that is NOT divisible by `k`. We also need to output the jumps themselves.

Let's analyze the possible scenarios:

1.  **Case 1: `x` is not divisible by `k` (`x % k != 0`)**
    If the target distance `x` itself is not divisible by `k`, the grasshopper can simply make a single jump of distance `x`. This jump is valid according to the rules, and it reaches the target in one move. This is clearly the minimum possible number of moves.

2.  **Case 2: `x` is divisible by `k` (`x % k == 0`)**
    In this case, a single jump of distance `x` is not allowed because `x % k == 0`. Therefore, at least two moves are required. We need to show that two moves are always sufficient.
    Consider making two jumps: `j1` and `j2`, such that `j1 + j2 = x`. We need both `j1 % k != 0` and `j2 % k != 0`.
    A simple strat

# Problem 10 Unit Array

In [73]:
state = {
    "problem": """
A. Unit Array

Given an array consisting only of -1 and 1.

A good array satisfies:
1. Sum of all elements >= 0
2. Product of all elements = 1

In one operation, you can change any element from -1 to 1 or from 1 to -1.

Find the minimum number of operations needed to make the array good.

Input:
t test cases.

For each test case:
n
a1 a2 ... an

Output:
Minimum operations for each test case.
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [74]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

Solved: True
import sys

def solve():
    n = int(sys.stdin.readline())
    a = list(map(int, sys.stdin.readline().split()))

    neg_count = 0
    for x in a:
        if x == -1:
            neg_count += 1
    
    pos_count = n - neg_count

    operations = 0

    # Phase 1: Satisfy Sum Condition (pos_count >= neg_count)
    # To increase the sum, we must change -1s to 1s.
    # Each such operation decreases neg_count by 1 and increases pos_count by 1.
    # This increases the total sum by 2.
    while neg_count > pos_count:
        neg_count -= 1  # Change a -1 to 1
        pos_count += 1
        operations += 1
    
    # After this loop, the sum condition (pos_count >= neg_count) is satisfied.
    # 'neg_count' now holds the number of -1s remaining after 'operations' changes.

    # Phase 2: Satisfy Product Condition (final neg_count must be even)
    # The product of all elements is 1 if and only if the number of -1s is even.
    # If the current 'neg_count' (after Phase 1 operat

# Problem 11 Forbidden Integer

In [75]:
state = {
    "problem": """
A. Forbidden Integer

You are given n, k and x.

You have unlimited copies of integers from 1 to k except x.

Determine whether it is possible to obtain sum n.

If possible, output:
YES
m
list of m integers

Otherwise output NO.

Input:
t test cases.

Each test case contains:
n k x

Output:
Construct any valid solution or print NO.
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [76]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

Solved: True
import sys

def solve():
    n, k, x = map(int, sys.stdin.readline().split())

    if x != 1:
        # If 1 is not forbidden, we can always make n by using n copies of 1.
        sys.stdout.write("YES\n")
        sys.stdout.write(str(n) + "\n")
        sys.stdout.write(" ".join(["1"] * n) + "\n")
    else:
        # If 1 is forbidden (x == 1)
        if k == 1:
            # Only 1 is available in [1, k], but 1 is forbidden. No numbers available.
            sys.stdout.write("NO\n")
        elif k == 2:
            # Only 2 is available in [1, k], as 1 is forbidden.
            if n % 2 == 0:
                # If n is even, we can use n/2 copies of 2.
                sys.stdout.write("YES\n")
                sys.stdout.write(str(n // 2) + "\n")
                sys.stdout.write(" ".join(["2"] * (n // 2)) + "\n")
            else:
                # If n is odd, we cannot make it with only 2s.
                sys.stdout.write("NO\n")
        else: # k >= 3
            # 2 an

Conclusion
Implemented a Competitive Programming Agent using LangGraph and Gemini API.
Workflow:
Analyze Problem → Generate Code → Evaluate Solution → Improve Solution
The agent was tested on multiple Codeforces CP-31 problems and successfully generated Python solutions automatically.